# 🎧 Playlist Creator: model training (Google Colab)

This notebook trains every model the app uses. **Nothing needs to be uploaded**: all audio is streamed from links.

| Part | What it does | Data (linked, not uploaded) | Time |
|---|---|---|---|
| 1 | Loads **YAMNet**, a CNN pretrained by Google on millions of YouTube clips. It is the app's "ears" | Hugging Face | 1 min |
| 2 | **Collects songs**: streams 30 s Deezer previews and turns each into 2,053 numbers | `dataset.csv` + `genre_groups.json` from GitHub | ~1–2 h (resumes) |
| 3 | **Genre**: LightGBM classifier for 19 broad genres (Reggaeton, Latin, House, K-pop, Rock…) | songs from Part 2 | ~15 min |
| 4 | **Mood**: LightGBM regressors for danceability, energy, valence, acousticness | songs from Part 2 | ~10 min |

**Why this architecture (transfer learning):** YAMNet has already learned what music *sounds like*, so we keep it frozen,
turn each song into 1,024 numbers (an *embedding*), and train only small, fast models on top.

**Genres:** Spotify's 114 genre labels in `dataset.csv` are grouped into 19 broad genres by `data/genre_groups.json`.
Labels that are moods or countries (happy, sad, british…) are left out. The genre is used as a filter for mood playlists
(e.g. "happy reggaeton"), so only the broad genres matter.

**Before you start:** `Runtime → Change runtime type → CPU` is fine (a GPU barely helps here). Run the cells top to bottom.
Progress is saved to your Google Drive, so if Colab disconnects, just re-run from the top and it resumes.

In [ ]:
!pip -q install onnxruntime lightgbm librosa pyarrow

In [ ]:
import io, json, os, tempfile, time, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

import lightgbm as lgb
import librosa
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import requests
import soundfile as sf

warnings.filterwarnings("ignore")
SR = 16000          # YAMNet expects 16 kHz mono audio
CLIP_SECONDS = 30   # previews and GTZAN clips are 30 s
SEED = 42

In [ ]:
# Everything this notebook produces (caches + final models) goes to Google Drive.
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/playlist-creator"
os.makedirs(OUT, exist_ok=True)
print("Saving to", OUT)

## Part 1: the "ears" (pretrained YAMNet)

YAMNet takes a waveform and, every 0.48 s, outputs a 1,024-number embedding describing the sound.
For a 30 s song we get ~62 embeddings; we summarize them with their **mean** (the overall sound) and **standard deviation** (how much it changes).

We also add 5 rhythm/loudness numbers from Librosa (tempo, loudness and beat strength), because danceability and energy depend on them.
**Total: 2,053 features per song.** The app will compute exactly the same features later.

In [ ]:
YAMNET_URL = "https://huggingface.co/audiomagic/yamnet-onnx/resolve/main/yamnet.onnx"  # Google's YAMNet, converted to ONNX
if not os.path.exists("yamnet.onnx"):
    with open("yamnet.onnx", "wb") as f:
        f.write(requests.get(YAMNET_URL, timeout=300).content)
yamnet = ort.InferenceSession("yamnet.onnx", providers=["CPUExecutionProvider"])

EXTRA_NAMES = ["tempo", "rms_mean", "rms_std", "onset_mean", "pulse_clarity"]
FEATURE_NAMES = ([f"emb_mean_{i}" for i in range(1024)] + [f"emb_std_{i}" for i in range(1024)] + EXTRA_NAMES)


def extract_features(y):
    # y: mono float32 waveform at 16 kHz, values in [-1, 1]. Returns 2,053 numbers.
    y = np.asarray(y, dtype=np.float32)[: SR * CLIP_SECONDS]
    _, emb, _ = yamnet.run(None, {"waveform": y})
    onset = librosa.onset.onset_strength(y=y, sr=SR)
    tempo, beats = librosa.beat.beat_track(onset_envelope=onset, sr=SR)
    tempo = float(np.atleast_1d(tempo)[0])
    pulse = float(onset[beats].mean() / (onset.mean() + 1e-8)) if len(beats) else 0.0
    rms = librosa.feature.rms(y=y)[0]
    extras = np.array([tempo, rms.mean(), rms.std(), onset.mean(), pulse], dtype=np.float32)
    return np.nan_to_num(np.concatenate([emb.mean(0), emb.std(0), extras]).astype(np.float32))


# Sanity check on 3 s of a test tone
print(len(FEATURE_NAMES), extract_features(np.sin(np.linspace(0, 3000, SR * 3)).astype(np.float32)).shape)

## Part 2: collect songs

`dataset.csv` has ~114k songs with Spotify's genre label and original mood scores (the **labels** the models learn from).
For each chosen song we search Deezer, stream its 30 s preview, and compute the same 2,053 features the app computes.

Two samples share one cache on Drive (`mood_features.parquet`, the same file earlier runs created, so nothing is redone):
1. **Mood sample:** 45 songs per Spotify genre (~5,000), as before.
2. **Genre top-up:** extra songs until every broad genre has ~250 songs with a preview, at most 3 per artist.

A song listed under Spotify genres from *different* broad genres (e.g. both "latin" and "pop") gets no genre label;
it is still used for mood.

In [ ]:
REPO_RAW = "https://raw.githubusercontent.com/Edwin-Trejo/Automated-playlist-creator/main"
TARGETS = ["danceability", "energy", "valence", "acousticness"]
PER_GENRE = 45        # mood sample: songs per Spotify genre
GENRE_TARGET = 250    # genre training: songs with a preview per broad genre
MAX_PER_ARTIST = 3

GENRE_GROUPS = requests.get(f"{REPO_RAW}/data/genre_groups.json", timeout=30).json()
SUB_TO_GENRE = {sub: g for g, subs in GENRE_GROUPS.items() for sub in subs}
GENRES = sorted(GENRE_GROUPS)

songs = pd.read_csv(f"{REPO_RAW}/data/dataset.csv").dropna(subset=["track_name", "artists"])
broad = songs.assign(genre=songs["track_genre"].map(SUB_TO_GENRE)).dropna(subset=["genre"]).groupby("track_id")["genre"].unique()
BROAD = broad[broad.str.len() == 1].str[0]  # track_id -> broad genre (only when all its Spotify genres agree)

songs = songs.drop_duplicates("track_id").reset_index(drop=True)
songs["artist"] = songs["artists"].str.split(";").str[0]
songs["genre"] = songs["track_id"].map(BROAD)
mood_sample = songs.sample(frac=1, random_state=SEED).groupby("track_genre").head(PER_GENRE).reset_index(drop=True)
print(f"{len(songs)} songs in dataset.csv, {songs['genre'].notna().sum()} with a broad genre ({len(GENRES)} genres)")
print(f"mood sample: {len(mood_sample)} songs")
songs["genre"].value_counts()

In [ ]:
DEEZER = "https://api.deezer.com/search"


def deezer_search(query):
    for attempt in range(4):
        try:
            data = requests.get(DEEZER, params={"q": query}, timeout=10).json()
        except Exception:
            time.sleep(2 * (attempt + 1))
            continue
        if (data.get("error") or {}).get("code") == 4:  # Deezer rate limit: wait and retry
            time.sleep(2 * (attempt + 1))
            continue
        return data.get("data", [])
    return []


def find_preview(track, artist):
    track, artist = track.replace('"', ""), artist.replace('"', "")
    for item in deezer_search(f'artist:"{artist}" track:"{track}"'):
        if item.get("preview"):
            return item["preview"]
    wanted = artist.lower()  # loose search: only accept results by the same artist
    for item in deezer_search(f"{track} {artist}"):
        found = (item.get("artist") or {}).get("name", "").lower()
        if item.get("preview") and found and (found in wanted or wanted in found):
            return item["preview"]
    return None


def load_preview(url):
    data = requests.get(url, timeout=15).content
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
        f.write(data)
    try:
        y, _ = librosa.load(f.name, sr=SR, mono=True, duration=CLIP_SECONDS)
    finally:
        os.remove(f.name)
    return y if len(y) and np.abs(y).max() > 0 else None


def process(row):
    try:
        artist = str(row["artists"]).split(";")[0]
        url = find_preview(str(row["track_name"]), artist)
        y = load_preview(url) if url else None
    except Exception:
        y = None
    if y is None:
        return {"track_id": row["track_id"], "ok": False}
    return {"track_id": row["track_id"], "ok": True, "artist": artist, "track_genre": row["track_genre"],
            **{t: float(row[t]) for t in TARGETS}, **dict(zip(FEATURE_NAMES, extract_features(y)))}


r = process(mood_sample.iloc[0].to_dict())
print("test song:", mood_sample.iloc[0]["track_name"], "->", "ok" if r["ok"] else "no preview found")

In [ ]:
FEATURE_CACHE, FAILED_CACHE = f"{OUT}/mood_features.parquet", f"{OUT}/mood_failed.csv"  # names kept so old runs are reused


def save_progress(rows, failed):
    if rows:
        new = pd.DataFrame(rows)
        if os.path.exists(FEATURE_CACHE):
            new = pd.concat([pd.read_parquet(FEATURE_CACHE), new], ignore_index=True)
        new.to_parquet(FEATURE_CACHE, index=False)
    if failed:
        pd.DataFrame({"track_id": failed}).to_csv(FAILED_CACHE, mode="a", header=not os.path.exists(FAILED_CACHE), index=False)


def already_tried():
    done = set(pd.read_parquet(FEATURE_CACHE)["track_id"]) if os.path.exists(FEATURE_CACHE) else set()
    if os.path.exists(FAILED_CACHE):
        done |= set(pd.read_csv(FAILED_CACHE)["track_id"])
    return done


def collect(todo):
    rows, failed, start = [], [], time.time()
    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = [pool.submit(process, r) for r in todo]
        for i, fut in enumerate(as_completed(futures), 1):
            res = fut.result()
            if res["ok"]:
                rows.append(res)
            else:
                failed.append(res["track_id"])
            if i % 200 == 0 or i == len(todo):
                save_progress(rows, failed)
                rows, failed = [], []
                rate = i / (time.time() - start)
                print(f"{i}/{len(todo)} songs | {rate:.1f}/s | ~{(len(todo) - i) / rate / 60:.0f} min left")


done = already_tried()
todo = [r for r in mood_sample.to_dict("records") if r["track_id"] not in done]
print(f"Mood sample: {len(done)} already processed, {len(todo)} to go")
collect(todo)

### Genre top-up
Count the songs each broad genre already has, then try more until it reaches ~250. About 1 in 5 songs has no Deezer
preview, so each round tries a few extra and up to 3 rounds fill the gaps.

In [ ]:
def genre_counts():
    cache = pd.read_parquet(FEATURE_CACHE)
    return cache.assign(genre=cache["track_id"].map(BROAD)).dropna(subset=["genre"])


for round_no in range(1, 4):
    have, skip = genre_counts(), already_tried()
    counts = have["genre"].value_counts()
    per_artist = have.groupby(["genre", "artist"]).size().to_dict()
    todo = []
    for g in GENRES:
        need = GENRE_TARGET - counts.get(g, 0)
        if need <= 0:
            continue
        pool = songs[(songs["genre"] == g) & ~songs["track_id"].isin(skip)].sample(frac=1, random_state=SEED + round_no)
        picked = []
        for r in pool.to_dict("records"):
            if len(picked) >= int(need * 1.3) + 5:
                break
            if per_artist.get((g, r["artist"]), 0) < MAX_PER_ARTIST:
                per_artist[(g, r["artist"])] = per_artist.get((g, r["artist"]), 0) + 1
                picked.append(r)
        todo += picked
    print(f"Round {round_no}: {len(todo)} songs to try")
    if not todo:
        break
    collect(todo)

print(genre_counts()["genre"].value_counts())

## Part 3: genre (19 broad genres)

**Fair evaluation:** many songs share an artist. If an artist appears in both training and test data, the model can
"recognize the artist" instead of the genre, which inflates accuracy. So we split **by artist**: no artist appears on both sides.
We report accuracy and **macro-F1** (the average score over genres, so small genres count as much as big ones).

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

spot = genre_counts().reset_index(drop=True)
X = spot[FEATURE_NAMES].to_numpy(np.float32)
y = spot["genre"].map(GENRES.index).to_numpy()
groups = spot["artist"].to_numpy()


def artist_split(idx, seed):
    # Split the rows `idx` ~80/20 so that no artist is on both sides.
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(X[idx], y[idx], groups[idx]))
    return idx[a], idx[b]


train_idx, test_idx = artist_split(np.arange(len(y)), SEED)
fit_idx, val_idx = artist_split(train_idx, SEED + 1)  # validation set for early stopping (test stays untouched)
assert not set(groups[train_idx]) & set(groups[test_idx])
print(f"train {len(fit_idx)} | validation {len(val_idx)} | test {len(test_idx)}")

### Optional: GTZAN as extra training songs
If an earlier run cached GTZAN's features on your Drive, its 999 clips are mapped to the broad genres and added to the
**training** data only (the test set stays Spotify songs). Both versions are trained below and the better one is kept.
Nothing is downloaded if the cache isn't there.

In [ ]:
GTZAN_CACHE = f"{OUT}/gtzan_features.parquet"
GTZAN_TO_GENRE = {"blues": "Jazz / Blues", "jazz": "Jazz / Blues", "classical": "Classical", "country": "Country / Folk",
                  "disco": "R&B / Soul / Funk", "hiphop": "Hip-Hop", "metal": "Metal", "pop": "Pop",
                  "reggae": "Reggae / Dancehall", "rock": "Rock"}
if os.path.exists(GTZAN_CACHE):
    gtzan = pd.read_parquet(GTZAN_CACHE)
    Xg = gtzan[FEATURE_NAMES].to_numpy(np.float32)
    yg = gtzan["genre"].map(GTZAN_TO_GENRE).map(GENRES.index).to_numpy()
    print(f"GTZAN: {len(Xg)} extra training clips")
else:
    Xg, yg = np.empty((0, X.shape[1]), np.float32), np.empty(0, int)
    print("No GTZAN cache on Drive: training on Spotify songs only")

### LightGBM on the embeddings
Gradient-boosted decision trees. `colsample_bytree=0.3` makes each tree look at a random 30% of the 2,053 features,
and `class_weight="balanced"` stops big genres from drowning out small ones.

In [ ]:
from sklearn.metrics import f1_score

LGBM_GENRE_PARAMS = dict(n_estimators=2000, learning_rate=0.03, num_leaves=15, colsample_bytree=0.3, subsample=0.8,
                         subsample_freq=1, class_weight="balanced", random_state=SEED, verbose=-1)

variants = {"spotify": (X[fit_idx], y[fit_idx])}
if len(Xg):
    variants["spotify+gtzan"] = (np.vstack([X[fit_idx], Xg]), np.concatenate([y[fit_idx], yg]))

genre_results, test_preds = {}, {}
for name, (Xf, yf) in variants.items():
    m = lgb.LGBMClassifier(**LGBM_GENRE_PARAMS)
    m.fit(Xf, yf, eval_set=[(X[val_idx], y[val_idx])], callbacks=[lgb.early_stopping(100, verbose=False)])
    test_preds[name] = m.predict(X[test_idx])
    genre_results[name] = {"accuracy": float((test_preds[name] == y[test_idx]).mean()),
                           "macro_f1": float(f1_score(y[test_idx], test_preds[name], average="macro")),
                           "trees": int(m.best_iteration_)}
    print(name, genre_results[name])

best = max(genre_results, key=lambda n: genre_results[n]["macro_f1"])
print("Keeping:", best)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

print(classification_report(y[test_idx], test_preds[best], labels=range(len(GENRES)), target_names=GENRES, zero_division=0))
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(y[test_idx], test_preds[best], labels=range(len(GENRES)), display_labels=GENRES,
                                        xticks_rotation=90, ax=ax, colorbar=False)
plt.title("Genre: test set (unseen artists)"); plt.tight_layout(); plt.show()

### Final genre model: retrain on all songs
Evaluation is done. The model we ship uses all the data, keeping the tree count found above.

In [ ]:
X_all, y_all = (np.vstack([X, Xg]), np.concatenate([y, yg])) if best == "spotify+gtzan" else (X, y)
final_lgbm = lgb.LGBMClassifier(**{**LGBM_GENRE_PARAMS, "n_estimators": genre_results[best]["trees"]}).fit(X_all, y_all)
final_lgbm.booster_.save_model(f"{OUT}/genre_lgbm.txt")
with open(f"{OUT}/genre_meta.json", "w") as f:
    json.dump({"genres": GENRES, "groups": GENRE_GROUPS, "test_accuracy": genre_results, "chosen": best, "songs": len(y_all),
               "feature_names": FEATURE_NAMES, "sample_rate": SR, "clip_seconds": CLIP_SECONDS}, f)
print("Saved genre model to", OUT)

## Part 4: mood (danceability, energy, valence, acousticness)

Every song collected in Part 2 has Spotify's original mood scores, so all of them are used here, including the genre top-up songs.
Same idea as the genre split: songs by the same artist stay on the same side, so the test score reflects **new** artists.
We compare each model against a baseline that always predicts the average; a model is only useful if it clearly beats it.

In [ ]:
mood = pd.read_parquet(FEATURE_CACHE)
print(f"Songs for mood training: {len(mood)}")

from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

Xm = mood[FEATURE_NAMES].to_numpy(np.float32)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(Xm, groups=mood["artist"]))
LGBM_MOOD_PARAMS = dict(n_estimators=3000, learning_rate=0.02, num_leaves=31, colsample_bytree=0.3,
                        subsample=0.8, subsample_freq=1, min_child_samples=20, random_state=SEED, verbose=-1)

mood_results, mood_trees, preds = {}, {}, {}
for t in TARGETS:
    yt = mood[t].to_numpy()
    inner_tr, inner_val = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
                               .split(Xm[tr], groups=mood["artist"].iloc[tr]))
    m = lgb.LGBMRegressor(**LGBM_MOOD_PARAMS)
    m.fit(Xm[tr][inner_tr], yt[tr][inner_tr], eval_set=[(Xm[tr][inner_val], yt[tr][inner_val])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    preds[t] = np.clip(m.predict(Xm[te]), 0, 1)
    mood_trees[t] = m.best_iteration_
    mood_results[t] = {"mae": mean_absolute_error(yt[te], preds[t]),
                       "baseline_mae": mean_absolute_error(yt[te], np.full(len(te), yt[tr].mean())),
                       "r2": r2_score(yt[te], preds[t])}

results = pd.DataFrame(mood_results).T.round(3)
results["previous (Librosa-only, local)"] = {"danceability": 0.612, "energy": 0.737, "valence": 0.531, "acousticness": 0.691}
print("r2 = share of variation explained (1.0 = perfect); previous column is R2 from the first model")
results

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, t in zip(axes, TARGETS):
    ax.scatter(mood[t].to_numpy()[te], preds[t], s=4, alpha=0.4)
    ax.plot([0, 1], [0, 1], "r--")
    ax.set_title(f"{t} (R2 {mood_results[t]['r2']:.2f})"); ax.set_xlabel("Spotify value"); ax.set_ylabel("predicted")
plt.tight_layout(); plt.show()

### Final mood models: retrain on all songs and save

In [ ]:
for t in TARGETS:
    final = lgb.LGBMRegressor(**{**LGBM_MOOD_PARAMS, "n_estimators": mood_trees[t]}).fit(Xm, mood[t].to_numpy())
    final.booster_.save_model(f"{OUT}/mood_{t}.txt")
with open(f"{OUT}/mood_meta.json", "w") as f:
    json.dump({"targets": TARGETS, "test_results": mood_results, "songs": len(mood),
               "feature_names": FEATURE_NAMES, "sample_rate": SR, "clip_seconds": CLIP_SECONDS}, f)
print("Saved mood models to", OUT)

## Done: download the models

The cell below zips the final models (not the large feature caches). Unzip it into the repo's `models/` folder;
connecting them to the app is the next step we'll do together.

In [ ]:
import shutil
bundle = "/content/playlist_models"
os.makedirs(bundle, exist_ok=True)
for name in ["genre_lgbm.txt", "genre_meta.json", "mood_meta.json"] + [f"mood_{t}.txt" for t in TARGETS]:
    shutil.copy(f"{OUT}/{name}", bundle)
shutil.copy("yamnet.onnx", bundle)
shutil.make_archive("/content/playlist_models", "zip", bundle)
from google.colab import files
files.download("/content/playlist_models.zip")